# 05 — 工具呼叫深入：`bind_tools`、`ToolNode`、`tools_condition`

**這份要學什麼**
- `bind_tools` / `ToolNode` / `tools_condition` 怎麼串成一個完整的工具呼叫迴圈
- 陷阱：`ToolNode` 預設不會接住工具「執行時」丟出的例外
- `return_direct=True`：只有 `create_agent` 認得，手刻版本不會理它

> 不需要 API key：一樣用 `scripted_model()` 走完整條 ReAct 迴圈，包括一次呼叫多個工具、以及一個容易踩到的錯誤處理陷阱。

`02_three_agents_compare.ipynb` 已經看過 `StateGraph + ToolNode + tools_condition` 這個組合，這份 notebook 拆開來看每一部分實際在做什麼。

## Agent 是什麼？想像一位會打電話查資料的客服

一般的 LLM 只會「聊天」——你問它「台北現在幾點」，它答不出來，因為它不知道現在的時間，也沒辦法自己去查。

Agent 的做法，是讓這位 LLM 客服人員多一個能力：遇到自己答不出來的問題，就「打電話給後台系統」查資料，查到結果再回來組織語言回答你。這個來回不是只發生一次——LLM 可能查了天氣之後還想再查一次時間，直到手上的資料夠回答問題，才會真正開口回答。這其實就是 `04_langgraph_control_flow.ipynb` 學到的迴圈，只是這次繞回去的原因變成了「LLM 還想再查一次工具結果」。

流程長這樣：

```
                 工具結果送回去，LLM 再想一次
        ┌───────────────────────────────────┐
        │                                   │
        ▼                                   │
   [ agent：LLM ] ──有 tool_calls──▶ [ tools：ToolNode ]
        │
        └──沒有 tool_calls──▶ END（直接回答你）
```

決定走哪條路的，永遠是 `agent` 這一站的 LLM 輸出，`tools` 這一站只負責執行、不做決定：

- **`bind_tools`**：先告訴這位客服人員「你有哪些電話可以打」（有哪些工具、參數長怎樣）
- **`ToolNode`**：客服人員真的拿起電話查資料的動作，查完把結果記下來
- **`tools_condition`**：站在電話亭門口的保全，看客服人員這次有沒有要打電話，決定放去 `ToolNode` 還是直接結束

下面 `@tool` 裝飾器定義的函式，它的 docstring 會變成「這支電話能查什麼」的說明，一起送給模型參考。

In [1]:
import sys

sys.path.insert(0, ".")
from _llm import scripted_model

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: sunny, 28C"


@tool
def get_local_time(city: str) -> str:
    """Get the current local time for a city."""
    return f"{city}: 14:00"


TOOLS = [get_weather, get_local_time]

## `bind_tools`：先告訴 LLM「你有哪些電話可以打」

`model.bind_tools(tools)` 回傳一個新的 Runnable，之後每次 `.invoke()` 都會把這些工具的名字、要傳什麼參數（schema）一起送給模型——模型才知道「有哪些電話號碼、每支電話要報什麼資料」。真正的模型會依此自己判斷要不要在回應裡帶上 `tool_calls`；這裡用的 `scripted_model` 不會真的讀 schema，而是照劇本回應——但 `bind_tools` 這個動作本身、以及後面 `ToolNode` 依 `tool_calls` 真的去執行工具的流程，跟接真模型時完全一樣。

## 一次問兩個問題：一則 `AIMessage` 帶多個 `tool_calls`

客服人員不一定一次只打一通電話——如果同時想知道天氣又想知道時間，`tool_calls` 列表裡就會有兩筆紀錄，`ToolNode` 會一次把兩通電話都打完，各自產生一則 `ToolMessage`（用 `tool_call_id` 對應回原本是哪一通電話問的）。

In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from _graph_viz import show_graph

model = scripted_model(
    [
        AIMessage(
            content="",
            tool_calls=[
                {"name": "get_weather", "args": {"city": "Taipei"}, "id": "c1"},
                {"name": "get_local_time", "args": {"city": "Taipei"}, "id": "c2"},
            ],
        ),
        AIMessage(content="台北現在 14:00，天氣晴朗 28 度。"),
    ]
).bind_tools(TOOLS)


def call_model(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")
agent = builder.compile()

show_graph(agent)

result = agent.invoke({"messages": [HumanMessage("台北現在幾點？天氣如何？")]})
for m in result["messages"]:
    print(f"{type(m).__name__:14} {m.content!r:45} tool_calls={getattr(m, 'tool_calls', None)}")

## `tools_condition` 內部邏輯：就是那個保全，規則很單純

`tools_condition` 做的事就是看「最後一則訊息有沒有 `tool_calls`」：有，就放你去 `"tools"`（去打電話）；沒有，就放你去 `"__end__"`（直接結束，回答完畢）。這正是 `04_langgraph_control_flow.ipynb` 手刻過的「保全站崗」條件邊模式，只是官方幫你包好了，還處理了幾種輸入型態（訊息列表 / dict / pydantic model）。

In [3]:
no_tool_call_state = {"messages": [AIMessage(content="純文字回答，沒有 tool_calls")]}
with_tool_call_state = {
    "messages": [AIMessage(content="", tool_calls=[{"name": "get_weather", "args": {"city": "Taipei"}, "id": "c1"}])]
}

print(tools_condition(no_tool_call_state))  # -> __end__
print(tools_condition(with_tool_call_state))  # -> tools

__end__
tools


## 陷阱：`ToolNode` 預設不會幫你接住工具「執行時」丟出的例外

這是這個版本（`langgraph 1.2`）容易搞錯的地方：`ToolNode` 預設的 `handle_tool_errors` 只處理「參數驗證錯誤」（模型傳的參數不符合工具要求的格式）。工具函式**自己執行時**丟出的例外（像下面的 `ZeroDivisionError`）預設會直接往外拋，整個 graph 的 `.invoke()` 會直接失敗，而不是讓客服人員優雅地說「查不到，我換個方式再試試」。

要讓工具執行期的錯誤也變成一則 `ToolMessage` 回饋給模型（讓模型有機會看到錯誤、自己修正做法），必須明確設定 `ToolNode(tools, handle_tool_errors=True)`。

In [4]:
@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


bad_call = AIMessage(content="", tool_calls=[{"name": "divide", "args": {"a": 1, "b": 0}, "id": "c1"}])

# 預設 handle_tool_errors：執行期例外會往外拋
default_builder = StateGraph(MessagesState)
default_builder.add_node("tools", ToolNode([divide]))
default_builder.add_edge(START, "tools")
default_builder.add_edge("tools", END)
default_graph = default_builder.compile()

try:
    default_graph.invoke({"messages": [bad_call]})
except ZeroDivisionError as e:
    print(f"預設行為：例外真的被拋出來了 -> {type(e).__name__}: {e}")

預設行為：例外真的被拋出來了 -> ZeroDivisionError: float division by zero


In [5]:
# 加上 handle_tool_errors=True：錯誤被包成 ToolMessage，模型還有機會看到錯誤訊息、修正做法
safe_builder = StateGraph(MessagesState)
safe_builder.add_node("tools", ToolNode([divide], handle_tool_errors=True))
safe_builder.add_edge(START, "tools")
safe_builder.add_edge("tools", END)
safe_graph = safe_builder.compile()

safe_result = safe_graph.invoke({"messages": [bad_call]})
print(safe_result["messages"][-1].content)

Error: ZeroDivisionError('float division by zero')
 Please fix your mistakes.


## `return_direct=True`：這通電話講完直接給客戶聽，不用主管轉述

正常流程是：工具（電話）講完 → 結果回報給 LLM（主管） → LLM 再組一句話講給使用者聽。
但有些工具本身的結果就是最終答案，不需要 LLM 再加工一次——`@tool(return_direct=True)`
標記的工具，執行完就直接把結果當成最後回覆，跳過「再問一次 LLM」這一步。

**陷阱**：這個旗標只有 `create_agent`（`02` 的 Tier B）自己組出來的圖會認得。我們手刻的
`tools_condition`（`02` 的 Tier C）完全不知道有這個欄位存在，工具跑完一樣乖乖繞回去問
LLM。下面直接對照兩種寫法的差異。

In [6]:
from langchain.agents import create_agent


@tool(return_direct=True)
def get_final_answer(question: str) -> str:
    """Return a canned final answer directly to the user."""
    return f"直接回答：{question} 的答案是 42"


# 用 create_agent：認得 return_direct，工具跑完直接結束
direct_model = scripted_model(
    [AIMessage(content="", tool_calls=[{"name": "get_final_answer", "args": {"question": "人生意義"}, "id": "c1"}])]
)
direct_agent = create_agent(model=direct_model, tools=[get_final_answer])
direct_result = direct_agent.invoke({"messages": [HumanMessage("人生意義是什麼")]})
print("用 create_agent：")
for m in direct_result["messages"]:
    print(f"  {type(m).__name__:12} {m.content!r}")

用 create_agent：
  HumanMessage '人生意義是什麼'
  AIMessage    ''
  ToolMessage  '直接回答：人生意義 的答案是 42'


In [7]:
# 手刻的 tools_condition：不認得 return_direct，工具跑完照樣繞回去問一次 LLM
manual_model = scripted_model(
    [
        AIMessage(content="", tool_calls=[{"name": "get_final_answer", "args": {"question": "人生意義"}, "id": "c1"}]),
        AIMessage(content="主管又轉述了一次：42。"),  # 這則不該出現，但手刻版本會多跑這一步
    ]
).bind_tools([get_final_answer])


def call_model(state: MessagesState) -> dict:
    return {"messages": [manual_model.invoke(state["messages"])]}


manual_builder = StateGraph(MessagesState)
manual_builder.add_node("agent", call_model)
manual_builder.add_node("tools", ToolNode([get_final_answer]))
manual_builder.add_edge(START, "agent")
manual_builder.add_conditional_edges("agent", tools_condition)
manual_builder.add_edge("tools", "agent")
manual_agent = manual_builder.compile()

manual_result = manual_agent.invoke({"messages": [HumanMessage("人生意義是什麼")]})
print("用手刻 tools_condition：")
for m in manual_result["messages"]:
    print(f"  {type(m).__name__:12} {m.content!r}")

用手刻 tools_condition：
  HumanMessage '人生意義是什麼'
  AIMessage    ''
  ToolMessage  '直接回答：人生意義 的答案是 42'
  AIMessage    '主管又轉述了一次：42。'


## 小結
- `bind_tools` 只是「告訴 LLM 有哪些電話可以打」，真的去打電話、把結果記回工作單是 `ToolNode` 的工作
- `tools_condition` 就是「看最後一則訊息有沒有 tool_calls」的保全站崗，沒有魔法
- 正式環境的工具**一定要**評估要不要加 `handle_tool_errors=True`（或自訂 callable），否則一次使用者輸入觸發的例外，會讓整個 agent 直接當機，而不是讓模型優雅地重試

下一份：`06_langgraph_memory_checkpoint.ipynb`，讓 agent 跨多輪對話記得之前說過什麼。